In [1]:
using Pkg; Pkg.status()

Status `~/.julia/dev/SymBoltz/presentation3/Project.toml`
  [d360d2e6] ChainRulesCore v1.26.1
  [6a86dc24] FiniteDiff v2.29.0
  [f6369f11] ForwardDiff v1.3.3
  [c9556dd2] ForwardDiffChainRules v0.3.0
⌃ [961ee093] ModelingToolkit v11.20.0
  [91a5bcdd] Plots v1.41.6
  [64c5a815] SymBoltz v1.4.1
Info Packages marked with ⌃ have new versions available and may be upgradable.


In [2]:
using Plots
Plots.default(framestyle = :box)

In [3]:
using ModelingToolkit

In [4]:
using SymBoltz

### 1) Model

In [5]:
M = ΛCDM(lmax = 16) # create symbolic model (can be modified)

Model ΛCDM: Standard cosmological constant and cold dark matter cosmological model
Subsystems (11): see hierarchy(ΛCDM)
  g: Spacetime FLRW metric in Newtonian gauge
  G: General relativity gravity
  γ: Photon radiation
  ν: Massless neutrinos
  c: Cold dark matter
  b: Baryonic matter
  ⋮
Equations (305):
  305 standard: see equations(ΛCDM)
Unknowns (224): see unknowns(ΛCDM)
  χ(τ): Conformal lookback time from today
  fν(τ): Neutrino-to-radiation density fraction
  ST(τ, k): Temperature source function
  ST_SW(τ, k): Sachs-Wolfe contribution to ST
  ST_ISW(τ, k): Integrated Sachs-Wolfe contribution to ST
  ST_Doppler(τ, k): Doppler contribution to ST
  ⋮
Parameters (31): see parameters(ΛCDM)
  C: Initial conditions integration constant
  τ0: Conformal time today
  τrec: Conformal time of recombination
  k: Perturbation mode wavenumber
  h: Dimensionless Hubble parameter today (H₀/(100km/s/Mpc))
  γ₊T₀: Temperature today (in K)
  ⋮

### 2) Problem

In [6]:
p = Dict(
    M.g.h => 0.70, M.b.Ω₀ => 0.05, M.c.Ω₀ => 0.25, M.ν.Neff => 3.0, M.h.m_eV => 0.02,
    M.γ.T₀ => 2.73, M.b.YHe => 0.25, M.h.N => 3.0, M.I.ln_As1e10 => 3.044, M.I.ns => 0.96,
)
prob = CosmologyProblem(M, p) # generate numerical code (can be customized)

Cosmology problem for model ΛCDM
Stages:
  Background: 5 unknowns, 0 splines, dense Jacobian
  Perturbations: 124 unknowns, 5 splines, 96.5 % sparse Jacobian
Parameters & initial conditions:
  ν₊Neff = 3.0
  h₊m_eV = 0.02
  b₊YHe = 0.25
  h₊N = 3.0
  I₊ln_As1e10 = 3.044
  h = 0.7
  γ₊T₀ = 2.73
  c₊Ω₀ = 0.25
  b₊Ω₀ = 0.05
  I₊ns = 0.96

### 3) Solution

In [9]:
ks = 10 .^ range(0, 3, length=400) # wavenumbers
sol = solve(prob, ks) # solve background and perturbations

Cosmology solution for model ΛCDM
Stages:
  Background: return code Terminated; solved with Rodas5P; 791 points
  Perturbations: return codes Success; solved with Rodas5P; 137-1896 points; x400 k ∈ [1.0, 1000.0] H₀/c (interpolation in-between)

In [15]:
@time Ps = spectrum_matter(prob, ks)
#jl = SphericalBesselCache(25:25:2500)
#Dls = spectrum_cmb([:TT, :TE, :EE], prob, jl; normalization = :Dl)

  0.721267 seconds (1.61 M allocations: 113.658 MiB)


400-element view(::Matrix{Float64}, 1, :) with eltype Float64:
 5.4754488525871715e-8
 5.5665555307919363e-8
 5.6591574944384745e-8
 5.753276227742389e-8
 5.848932726160227e-8
 5.946155433233948e-8
 6.044968766840556e-8
 6.145395760572287e-8
 6.247460713812775e-8
 6.351189560149783e-8
 6.456608173574045e-8
 6.563742570786854e-8
 6.672619146346542e-8
 ⋮
 2.572245042403731e-8
 2.49674516885129e-8
 2.41439630781371e-8
 2.3261478606248902e-8
 2.234469075849481e-8
 2.1427502696690097e-8
 2.0545334924613615e-8
 1.972758390127113e-8
 1.8991862241751228e-8
 1.8341096528570393e-8
 1.7763909731771004e-8
 1.723807350560629e-8

In [ ]:
plot(ls, Dls)

In [ ]:
equations(M.G)

In [ ]:
prob.pt.f.jac.f_iip

In [ ]:
p1 = plot(sol, log10(M.g.a), [log10(M.g.ℋ)])
p2 = plot(sol, log10(M.g.a), [M.b.Xe, M.b.rec.XH⁺, M.b.rec.XHe⁺⁺, M.b.rec.XHe⁺], legend = :left)
p3 = plot(sol, log10(M.g.a), [M.g.Φ, M.g.Ψ], [10.0, 100.0, 1000.0]; legend = :left)
p4 = plot(sol, M.g.z, [M.ST_Doppler, 10M.ST_SW, 100M.ST_ISW], [10.0, 100.0]; xlims = (0, 2300), legend = :bottomright, Nextra = 10)
plot(p1, p2, p3, p4, layout = (2, 2), size = (1000, 750), legendfontsize = 7)

## Plot recipes

In [ ]:
p = plot(sol, log10(M.g.a), [[M.γ.ρ, M.ν.ρ, M.h.ρ, M.c.ρ, M.b.ρ, M.Λ.ρ] ./ M.G.ρ; M.τ/M.τ0])
savefig(p, "media/bg.png")
p

In [ ]:
p = plot(sol, log10(M.g.a), [M.b.rec.XHe⁺⁺, M.b.rec.XHe⁺, M.b.rec.XH⁺, M.b.rei1.Xe, M.b.rei2.Xe, M.b.Xe], legend_position = :left)
savefig(p, "media/th.png")
p

In [ ]:
k = ks[end]
p = plot(sol, log10(M.g.a), log10.(abs.([M.γ.δ, M.ν.δ, M.h.δ, M.c.δ, M.b.δ])), k)
savefig(p, "media/pt.png")
p

In [ ]:
τs = sol.bg.t
τi = τs[begin]
τ0 = τs[end]
τs = collect(range(τi, τ0, length=8000))
τs[begin] = τi
τs[end] = τ0
lgas = sol(log10(M.g.a), τs)
S = sol(M.ST, τs, k)
kχ = k .* (τ0 .- τs)
l = 20
p = plot(τs, S .* SymBoltz.jl.(l, kχ), xlabel = "τ", label = "Sᵀ(τ,k) jₗ(k(τ₀-τ))")
lens!(p, [0.0, 0.1], [-0.002, +0.008], inset = (1, bbox(0.1, 0.0, 0.4, 0.4)), framestyle = :box, xticks = nothing, yticks = nothing, subplot = 2)
lens!(p, [2.8, 3.2], [-0.0007, +0.0007], inset = (1, bbox(0.6, 0.0, 0.4, 0.4)), framestyle = :box, xticks = nothing, yticks = nothing, subplot = 2)
savefig(p, "media/los.png")
p

In [ ]:
ks = 10 .^ range(0, 3, length = 200)
τs = range(sol.bg.t[end], 2.0, length=5)
zs = sol(M.g.z, τs) # corresponding redshifts
Ps = spectrum_matter(prob, ks, τs)

p = plot(log10.(ks), transpose(log10.(Ps)); xlabel = "log10(k)", label = permutedims("log10(P(k, z=".*string.(round.(abs.(zs);digits=1)) .* "))"), legend_position = :bottomleft)
savefig(p, "media/Pk.png")
p

In [ ]:
ls = 25:25:2500
jl = SphericalBesselCache(ls)
Dls = spectrum_cmb(:TT, prob, jl; normalization = :Dl)

p = plot(ls, Dls; xlabel = "l", label = "Dₗ")
savefig(p, "media/Dl.png")
p

## Features of symbolic-numeric interface

In [ ]:
equations(M.G)

In [ ]:
prob.pt.f.jac.f_iip

### With explicit solver

In [ ]:
k = 1.0
sol1 = solve(prob, k; ptopts = (alg = SymBoltz.Rodas5P(),))
sol2 = solve(prob, k; ptopts = (alg = SymBoltz.Tsit5(), maxiters = 1000))
p = plot(xlabel = "time", ylabel = "$(length(sol1.pts[1].prob.u0)) ODE variables")
plot!(p, sol1.pts[1]; color = :black, marker = :vline, label = permutedims(["implicit Rodas5P solver ($(length(sol1.pts[1].t)) points)"; fill(nothing, 100)]), denseplot = false)
plot!(p, sol2.pts[1]; color = :red,   marker = :vline, label = permutedims(["explicit Tsit5 solver ($(length(sol2.pts[1].t)) points)"; fill(nothing, 100)]), denseplot = false, xlims = extrema(sol1.pts[1].t))
savefig(p, "media/explicit.png")
p

## Autodiff

In [ ]:
using ForwardDiff, ChainRulesCore, ForwardDiffChainRules

function mytan(x)
    println("Hi it's me, tan($x)")
    return tan(x)
end
x₀ = π/4
ϵ = 1e-3
println("FD: tan′(π/4) ≈ ", (mytan(x₀+ϵ/2)-mytan(x₀-ϵ/2))/ϵ)

function ChainRulesCore.frule((_, dx), ::typeof(mytan), x)
    t = mytan(x) # value: tan(x)
    dt = 1 + t^2 # derivative: tan(x)′ = 1 + tan²(x)
    return t, dt * dx
end
@ForwardDiff_frule mytan(x::ForwardDiff.Dual)

println("AD: tan′(π/4) = ", ForwardDiff.derivative(mytan, x₀))

## Autodiff (v2)

In [ ]:
mysin(x) = (println("Hi it's me, sin($x)"); return sin(x))
mycos(x) = (println("Hi it's me, cos($x)"); return cos(x))
mytan(x) = (println("Hi it's me, tan($x)"); return mysin(x) / mycos(x))
f, x₀, ϵ = mytan, π/4, 1e-15
println("FD: tan′(π/4) ≈ ", (f(x₀+ϵ/2)-f(x₀-ϵ/2))/ϵ)

using ForwardDiff, ChainRulesCore, ForwardDiffChainRules
ChainRulesCore.frule((_, x′), ::typeof(mysin), x) = (mysin(x), +mycos(x) * x′) # (value, derivative)
ChainRulesCore.frule((_, x′), ::typeof(mycos), x) = (mycos(x), -mysin(x) * x′) # (value, derivative)
@ForwardDiff_frule mysin(x::ForwardDiff.Dual)
@ForwardDiff_frule mycos(x::ForwardDiff.Dual)
println("AD: tan′(π/4) = ", ForwardDiff.derivative(f, x₀))

In [ ]:
prob = CosmologyProblem(M, pars; jac = true, sparse = false)
vary = [M.c.Ω₀, M.b.Ω₀, M.ν.Neff, M.g.h, M.b.YHe, M.h.m_eV]
probgen = parameter_updater(prob, vary)
ks = 10 .^ range(0.5, 3.5, length=100)
P(k, θ) = spectrum_matter(probgen(θ), ks)
lgP(lgθ) = log10.(P(ks, 10 .^ lgθ)) # in log-space
θ = [0.27, 0.05, 3.0, 0.7, 0.25, 0.02]
#lgPs = lgP(log10.(θ))
@time dlgP_dlgθs_ad = ForwardDiff.jacobian(lgP, log10.(θ))
@time dlgP_dlgθs_fd = FiniteDiff.finite_difference_jacobian(lgP, log10.(θ))

In [ ]:
plot(log10.(ks), dlgP_dlgθs_ad; color = permutedims(eachindex(vary)), linestyle = :solid, label = nothing)
plot!(log10.(ks), dlgP_dlgθs_fd; color = permutedims(eachindex(vary)), linestyle = :dash, label = nothing)

## Jacobian sparsity

In [ ]:
J = Matrix(prob.pt.f.jac_prototype)
J .!= 0
#A = rand(2, 2)
heatmap(J)

In [ ]:
f1
f2 = f2(f1)
f3 = f3(f2)
f4 = f4(f3)
f5 = f5(f4)
...